In [16]:
import torch
import torch.nn.functional as F

In [17]:
def SIGReg(z, n_slices=16):
    #sigreg loss
    #each dim should be unit gaussian across the batch

    B, T, C = z.shape

    z = z.flatten(1) # BD

    slice_dirs = torch.randn((T*C, n_slices)) # DN
    slice_dirs = F.normalize(slice_dirs, dim=0) 

    slices = z@slice_dirs #BN

    t = torch.linspace(0, 5, 9) # frequencies, aka 't' : 9 points spaced evenly between 0 and 5

    sin_responses = torch.sin( t*slices[...,None] ).mean(dim=0) # t * BN1 -- broadcasts BN1 across t for BNt, then reduce B with mean
    cos_responses = torch.cos( t*slices[...,None] ).mean(dim=0) # t * BN1 -- broadcasts BN1 across t for BNt

    sin_target = 0
    cos_target = torch.exp(-0.5*t**2) # t
    weights = torch.exp(-0.5*t**2) # also t.  this weights the losses by a gaussian 

    sin_loss = F.mse_loss(sin_responses, sin_target)*weights # BNt 1   * t -> BNt
    cos_loss = F.mse_loss(cos_responses, cos_target)*weights # BNt t   * t -> BNt

    SIGReg_loss = 0.5*(sin_loss.mean() + cos_loss.mean())

    return SIGReg_loss

    

    

In [35]:
a = torch.randn(24, 32)

In [36]:
#a=F.normalize(a, dim=0)
a.norm(dim=0)

tensor([4.5509, 4.1641, 4.2329, 4.1842, 4.7858, 5.7063, 5.8613, 4.4945, 4.0399,
        4.8793, 5.7752, 4.4157, 6.0779, 5.1322, 4.7948, 5.7748, 5.5592, 5.6353,
        4.7970, 3.7192, 6.1953, 4.9601, 4.2262, 6.8214, 4.6750, 4.0416, 4.1603,
        3.5380, 4.9864, 4.5295, 4.0771, 5.1510])